In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from PIL import Image
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from timm.scheduler import CosineLRScheduler
import copy
from models import *
from utils import *
import albumentations as A
from dataset import CellDataset
import torchio as tio

from metrics import *
from vis_utils_3d import *

In [ ]:
TRAIN_PATH = 'INSERT PATH'
IS_3D = False

if IS_3D:
    dilation_iterations = 2
    
else:
    dilation_iterations = 1


BASE_PATH = os.getcwd()
GENERATE_TEST = True
is_trinary = True
num_workers = 22
# possible metrics = ['ASD', 'hausdorff', 'seg', '3d_seg']
metric_name = 'hausdorff'

generate_test_iterations = 1
GENERATE_TEST_PATH = os.path.join(os.getcwd(), 'datasets', 'data', 'revisions', metric_name)
LIST_VALID_PATH = os.path.join(BASE_PATH, 'datasets', 'data', 'revisions', metric_name, 'valid')
LIST_TEST_PATH = os.path.join(BASE_PATH, 'datasets', 'data', 'revisions', metric_name, 'test')


In [ ]:
print_dir_tree(TRAIN_PATH)

In [ ]:
images_paths, seg_paths = [], []
for name in ['0' + str(i) if i < 10 else str(i) for i in range(1, 45)]:
    try:
        curr_images, curr_labels = get_images_and_masks(name, TRAIN_PATH)
        if len(curr_images) == len(curr_labels):
            images_paths += curr_images
            seg_paths += curr_labels
    except:
        print('Finished')
        break

In [ ]:
assert len(images_paths) == len(seg_paths)

In [ ]:
print(f'Number of images in dataset = {len(images_paths)}')

In [ ]:
train_seg_paths, valid_seg_paths, train_images_paths, valid_images_paths = train_test_split(seg_paths, images_paths, test_size=0.3, shuffle=True, random_state=42)

#### In addition to Morphological Operations and Non-Rigid Perturbations we want to further enrich the variability of the dataset using augmentations

In [ ]:
if IS_3D:
    transform = tio.Compose([
        tio.CropOrPad((32, 256, 256)),  # Adjust the size for 3D cropping
        tio.RandomFlip(axes=(0,), p=0.5),  # Horizontal flip along one axis
        tio.RandomFlip(axes=(1,), p=0.5),  # Vertical flip along another axis
    ])

else:
    transform = A.Compose([
        A.CropNonEmptyMaskIfExists(256, 256),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
    ])

device = 'cuda'
metric = Metrics(metric_name)

In [ ]:
train_dataset = CellDataset(train_seg_paths,
                            train_images_paths,
                            metric,
                            transform,
                            is_trinary=is_trinary,
                            is_3d=IS_3D,
                            dilation_iterations=dilation_iterations)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=num_workers)

#### Generate a validation and test set based on the dataset, augmentations, morphological operations, non-rigid perturbations and metric we are using

In [ ]:
if GENERATE_TEST:
    print('generating data....')
    generate_train_test_data(valid_seg_paths,
                             valid_images_paths,
                             GENERATE_TEST_PATH,
                             is_trinary,
                             transform,
                             metric,
                             num_workers=num_workers,
                            is_3d=IS_3D,
                            num_epochs=generate_test_iterations,
                            dilation_iterations=dilation_iterations)

In [ ]:
valid_images, valid_segmentations, valid_labels = list_dataset_files(LIST_VALID_PATH)

test_images, test_segmentations, test_labels = list_dataset_files(LIST_TEST_PATH)

In [ ]:
for t in [valid_images, valid_segmentations, valid_labels, test_images, test_segmentations, test_labels]:
    t.sort()

In [ ]:
valid_dataset = CellDataset(valid_segmentations, valid_images, metric, train=False, is_trinary=is_trinary, label_dir=valid_labels, is_3d=IS_3D)
test_dataset = CellDataset(test_segmentations, test_images, metric, train=False, is_trinary=is_trinary, label_dir=test_labels, is_3d=IS_3D)


valid_dataloader = DataLoader(valid_dataset, batch_size=2, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=2, shuffle=True)

#### Sanity check

In [ ]:
# Checking the dataloader works
image, transformed_seg, labels, gt_seg = next(iter(train_dataloader))

print(image.shape, transformed_seg.shape, labels.shape)

In [ ]:
if IS_3D:
    plot3d_image_gt_pred_slices(image[1], gt_seg[1], transformed_seg[1], 6)
    print(labels[1])
else:
    num_images = 16
    fig, ax = plt.subplots(nrows=num_images, ncols=3, figsize=(20,80))
    for i in range(len(image)):
        if i > num_images - 1:
            break
        ax[i][0].imshow(image[i])
        ax[i][0].set_title('Cells Image')
        ax[i][1].imshow(transformed_seg[i])
        ax[i][1].set_title(f'{metric_name} = {round(labels[i].item(), 3)}')
        ax[i][2].imshow(gt_seg[i])
        ax[i][2].set_title('Ground Truth Segmentation')
    plt.show()

In [ ]:
rib_channels = [1, 32, 64, 128, 256]
model = RibCage(channels=rib_channels).to(device)


# channels_3d = [1, 64, 128, 256, 512]
# model = RibCage3D(channels=channels_3d).to(device)

# Ablation study
# naive_channels = [int(t * 2) for t in channels]
# model = Naive(channels=naive_channels).to(device)

# Ablation study
# siamese_channels = [1] + [int(t * 1.6) for t in [32, 64, 128, 256]]
# model = Siamese(channels=siamese_channels).to(device)


print(model)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

total_params = count_parameters(model)
print(f"Total trainable parameters: {total_params}")

# Training

In [ ]:
num_epochs = 100
criterion = nn.MSELoss(reduction='sum').to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
steps = num_epochs
warmup_steps = int(steps*0.1)
scheduler = CosineLRScheduler(optimizer, t_initial=steps, warmup_t=warmup_steps, warmup_lr_init=1e-6, lr_min=2e-8)
LOAD_MODEL = False

In [ ]:
if LOAD_MODEL:
    model_path = os.path.join('INSERT_PATH')
    model.load_state_dict(torch.load(model_path))
    print('Model loaded!')

In [ ]:
train_loss_history = []
val_loss_history = []
best_valid = float('inf')
model_path = os.path.join('INSERT_PATH')
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    num_examples = 0
    for image, transformed_seg, labels, _ in tqdm(train_dataloader):
        image, transformed_seg, labels = image.to(device), transformed_seg.to(device), labels.to(device)
        
        mask = (labels > 0)
        image, transformed_seg, labels = image[mask], transformed_seg[mask], labels[mask]
        if len(image) == 0:
            continue
        
        image, transformed_seg = image.unsqueeze(1), transformed_seg.unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(image.float(), transformed_seg.float())
        loss = criterion(outputs.flatten(), labels.flatten().float())
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        num_examples += image.size(0)

    epoch_loss = running_loss / num_examples

    train_loss_history.append(epoch_loss)
    scheduler.step(epoch)
    
    real_labels = []
    preds = []
    model.eval()
    with torch.no_grad():
        for image, transformed_seg, labels, _ in valid_dataloader:
            image, transformed_seg, labels = image.to(device).unsqueeze(1), transformed_seg.to(device).unsqueeze(1), labels.to(device)
            mask = (labels > 0)
            image, transformed_seg, labels = image[mask], transformed_seg[mask], labels[mask]
            if len(image) == 0:
                continue

            outputs = model(image.float(), transformed_seg.float())
            real_labels.append(labels)
            preds.append(outputs)

    preds = torch.cat(preds).flatten()
    real_labels = torch.cat(real_labels).flatten()

    err = criterion(preds, real_labels)
    
    err = err / len(preds)
    val_loss_history.append(err.item())


    if err < best_valid:
        best_model = copy.deepcopy(model)
        best_valid = err
        torch.save(best_model.state_dict(), model_path)
        print('Best new valid score')
    print(f'Epoch = {epoch + 1}/{num_epochs}, Loss = {epoch_loss}, Validation error = {err}')

In [ ]:
plt.plot(train_loss_history, label='Train Loss')
plt.plot(val_loss_history, label='Validation Loss')
plt.legend()
plt.title('Train VS Validation loss')

In [ ]:
real_labels = []
preds = []

keep = []
best_model.eval()
with torch.no_grad():
    for image, transformed_seg, labels, _ in tqdm(test_dataloader):
        image, transformed_seg = image.to(device), transformed_seg.to(device)
        mask = (labels > 0)
        image, transformed_seg, labels = image[mask], transformed_seg[mask], labels[mask]
        if len(image) == 0:
            continue
        outputs = best_model(image.float().unsqueeze(1), transformed_seg.unsqueeze(1).float())
        real_labels.append(labels)
        preds.append(outputs)

        labels = labels.to(device)
        mask = (torch.abs(labels - outputs.squeeze()) > 0.2)
        image, transformed_seg, labels, outputs = image[mask], transformed_seg[mask], labels[mask], outputs[mask]
        if len(image) != 0:
            keep.append([image, transformed_seg, labels, outputs])        


preds = torch.cat(preds).cpu().squeeze(1)
real_labels = torch.cat(real_labels).cpu()


errors = []


for i in range(len(preds)):
    error = criterion(preds[i], real_labels[i])
    errors.append(error)

errors = torch.tensor(errors)
print(f'Error rate = {errors.mean()}, STD = {errors.std()}')

In [ ]:
if IS_3D:
    image, transformed_seg, labels, outputs = keep[0]
    print(labels, outputs)
    plot3d_image_gt_pred_slices(image[0].cpu(), transformed_seg[0].cpu(), transformed_seg[0].cpu(), 7)

In [ ]:
from scipy import integrate

sns.set_style('darkgrid')

# Defining a range of absolute error tolerances
tolerances = np.linspace(0, 1, 50)

plt.figure(figsize=(5, 5))

hit_rates = [
    np.mean(np.array((np.abs(preds - real_labels) <= tol))) for tol in tolerances
]

auc = integrate.trapz(hit_rates, tolerances)
print(f'AUC Score = {auc}')

sns.lineplot(x=tolerances, y=hit_rates)

plt.xlabel('Absolute Error Tolerance')
plt.ylabel('Hit Rate')
plt.title('Hit Rate vs. Absolute Error Tolerance')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import seaborn as sns

sns.set_style('darkgrid')

limit = 1

preds_np = preds.cpu().numpy().flatten()
real_labels_np = real_labels.cpu().numpy().flatten()

joint_plot = sns.jointplot(x=real_labels_np, y=preds_np, kind="scatter", alpha=0.4, marginal_kws=dict(kde=True))

joint_plot.ax_joint.set_xlim(0, limit)
joint_plot.ax_joint.set_ylim(0, limit)

plt.plot([0, limit], [0, limit], 'p-')

plt.xlabel('GT Quality Measure')
plt.ylabel(f'Estimated Quality Measure')
plt.grid(visible=True)
plt.show()